# Body Measurement Regressor – Training
MaskAttentive Multi-View model: EfficientNet-B3 + segmentation attention.

In [ ]:
# ── 1. Mount Drive & install deps ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm

In [ ]:
# ── 2. Clone / upload your repo  ──────────────────────────────────────────
# Option A: clone from GitHub
# !git clone https://github.com/YOUR_USER/cc5-scripts.git /content/cc5-scripts

# Option B: copy ml/ folder from Drive (if you uploaded it there)
import shutil, os
shutil.copytree('/content/drive/MyDrive/cc5-scripts/ml', '/content/ml', dirs_exist_ok=True)

import sys
sys.path.insert(0, '/content')

In [ ]:
# ── 3. Unzip dataset (if stored as zip on Drive) ───────────────────────────
DRIVE_ZIP = '/content/drive/MyDrive/renders.zip'   # ← change path
DATA_ROOT = '/content/renders'

if not os.path.exists(DATA_ROOT):
    !unzip -q {DRIVE_ZIP} -d /content/

# Verify
!ls {DATA_ROOT}

In [ ]:
# ── 4. Configure & train ──────────────────────────────────────────────────
from ml.config import TrainConfig
from ml.train import train

cfg = TrainConfig()
cfg.data_root      = DATA_ROOT
cfg.checkpoint_dir = '/content/drive/MyDrive/checkpoints'  # saves to Drive
cfg.epochs         = 100
cfg.batch_size     = 8    # T4: 8 works fine with AMP; A100: try 16
cfg.img_size       = 256
cfg.num_workers    = 2    # Colab is usually happier with 2

history = train(cfg)

In [ ]:
# ── 5. Plot training curves ───────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open(f'{cfg.checkpoint_dir}/history.json') as f:
    history = json.load(f)

epochs   = [h['epoch'] for h in history]
val_mae  = [h['mean_mae_cm'] for h in history]
val_loss = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, val_mae);  ax1.set_title('Val MAE (cm)'); ax1.set_xlabel('Epoch')
ax2.plot(epochs, val_loss); ax2.set_title('Val Loss');     ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

# Per-measurement MAE at best epoch
best = min(history, key=lambda h: h['mean_mae_cm'])
from ml.config import MEASUREMENTS
print(f"\nBest epoch {best['epoch']} – MAE per measurement:")
for m in MEASUREMENTS:
    print(f'  {m:30s}: {best[m]:.2f} cm')

In [ ]:
# ── 6. Quick inference test ───────────────────────────────────────────────
import torch
from ml.model import build_model
from ml.config import MEASUREMENTS

device = torch.device('cuda')
ckpt = torch.load(f'{cfg.checkpoint_dir}/best.pt', map_location=device)

model = build_model(cfg).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

import json
with open(f'{cfg.checkpoint_dir}/norm_stats.json') as f:
    stats = json.load(f)
mean_arr = torch.tensor([stats['mean'][m] for m in MEASUREMENTS])
std_arr  = torch.tensor([stats['std'][m]  for m in MEASUREMENTS])

# Feed one sample from val set
from ml.dataset import build_datasets
_, val_ds = build_datasets(DATA_ROOT, cfg.img_size, cfg.val_split, cfg.seed)
sample = val_ds[0]

with torch.no_grad():
    preds_norm = model(
        sample['normal_sils'].unsqueeze(0).to(device),
        sample['masks'].unsqueeze(0).to(device),
    )[0].cpu()

preds_cm = preds_norm * std_arr + mean_arr
gt_cm    = sample['targets'] * std_arr + mean_arr

print(f"Character: {sample['char_id']}")
print(f"{'Measurement':30s}  {'GT':>8}  {'Pred':>8}  {'Err':>8}")
for i, m in enumerate(MEASUREMENTS):
    unit = 'L' if m == 'volume_L' else 'cm'
    print(f"{m:30s}  {gt_cm[i]:7.2f}{unit}  {preds_cm[i]:7.2f}{unit}  {abs(preds_cm[i]-gt_cm[i]):6.2f}")